![CCAI 9012 course banner](../../../docs/figs/brand_identity/banner.png)

# Controlled Image Generation and Caption Analysis

## Do AI-Generated Buildings Show Material Bias?

When a prompt does not specify materials, do image generators still repeatedly depict contemporary buildings as glass, concrete, or steel? Such patterns may reveal material biases or learned visual priors in how AI represents architecture.

This notebook tests that question with a small controlled experiment. We keep the prompt free of material terms, generate seed-based variations, and ask vision models to describe the apparent materials. Repeated labels can indicate systematic material associations, but they should not be treated as material ground truth or as representative estimates from only 50 images.

## What you will learn

By the end of this notebook, you should be able to:

- design a controlled prompt experiment with a fixed scene and changing seeds;
- distinguish a descriptive caption from a candidate material label;
- turn model answers into a small table; and
- read frequency and co-occurrence charts as patterns in model labels.

## Roadmap

This is a **text → image → text → labels → charts** experiment.

| Step | What you will do |
| --- | --- |
| 1 | Design a neutral-prompt, changing-seed experiment. |
| 2 | Generate a small set of building-image variations. |
| 3 | Ask BLIP and Qwen2.5-VL to turn images into text and material labels. |
| 4 | View frequency and co-occurrence patterns in the matched labels. |

<p align="center">
  <img src="../../../docs/figs/gen_image_eval_flow.svg" alt="A text-to-image-to-text flow. A neutral prompt is turned into generated building images with Stable Diffusion, then BLIP and Qwen-VL turn images into captions." width="100%"><br>
  <em>Figure 1. Start with a neutral prompt, generate images, then use captions to inspect what the models see.</em>
</p>

The flowchart combines both model directions: Stable Diffusion is the **text → image** part, while BLIP and Qwen2.5-VL are the **image → text** part. Their captions and answers give you language to compare with the original prompt before moving to the label analysis.

In [ ]:
from ccai9012.sd_utils import SDClient
from ccai9012.multi_modal_utils import VisionQAProcessor
from ccai9012 import viz_utils
from ccai9012.paths import MODELS_DIR, STARTER_KITS_DIR, ensure_output_dir

EXAMPLE_DIR = STARTER_KITS_DIR / '3_multimodal_reasoning' / 'gen_images_eval'
GENERATED_IMAGES_DIR = EXAMPLE_DIR / 'gen_imgs'
TEST_IMAGES_DIR = EXAMPLE_DIR / 'test_imgs'
OUTPUT_DIR = ensure_output_dir(EXAMPLE_DIR)

In [ ]:
import pandas as pd

## Step 1 — Design a controlled prompt experiment

<img src="../../../docs/figs/image_workflow_step_1.svg" alt="Four-stage image-material analysis workflow with Step 1, design the experiment, highlighted." width="100%">

The experiment needs one stable written scene. The next two cells create a **Stable Diffusion** text-to-image generator and make one image. The `model_id` chooses the generator, `mode` chooses how it runs, and `cache_dir` is where a local model is stored. First, simply inspect the image: which material words would *you* use to describe it?

A seed is the starting point for the generator's randomness. Keeping the prompt and seed fixed specifies the same experiment; changing the seed makes a controlled variation. In shorthand: `image = G(prompt, z_seed)`. Different hardware or software versions can still produce different pixels. For background, see [Week 4: Multimodal LLM tutorial](../../../weekly_scripts/wip/week4/week4_t_multimodal_llm.ipynb).

In [ ]:
# initialize Stable Diffusion Client
client = SDClient(
    mode="local",
    model_id="CompVis/stable-diffusion-v1-4",
    cache_dir=str(MODELS_DIR),
    device="cpu",
)

In [ ]:
prompt = "a futuristic art museum in a modern city with glass structure, daylight, exterior view"
images = client.generate_images(prompt=prompt, num_images=1, seed=42)

## Step 2 — Generate a seed-varied image set

<img src="../../../docs/figs/image_workflow_step_2.svg" alt="Four-stage image-material analysis workflow with Step 2, generate image variations, highlighted." width="100%">

One image is only one draw from the generator. The next cell uses the neutral prompt `a contemporary building, exterior view, urban context, daylight`—notice that it does not request glass, concrete, or any other material. Run it to save 50 images in `gen_imgs/`, keeping the prompt fixed while varying the seed.

This is a small **controlled prompt experiment**: the seed changes the initial noise, while the stated scene remains fixed. Before continuing, predict: will the buildings share obvious material cues, or will seed variation dominate?

In [ ]:
from tqdm import tqdm
from PIL import Image

# Define output folder to save generated images
GENERATED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Use a neutral and consistent prompt without material-related keywords
prompt = "a contemporary building, exterior view, urban context, daylight"

num_images = 50

# Generate 50 images with different random seeds
for i in tqdm(range(num_images), desc="Generating images"):
    
    # Generate one image per iteration with a unique seed
    images = client.generate_images(
        prompt=prompt,
        num_images=1,
        seed=i,  # Different seed to ensure variation
        display_images=False
    )

    # Save the generated image to disk
    img = images[0]
    img_path = GENERATED_IMAGES_DIR / f"building_exterior_{i+1:03d}.png"
    img.save(img_path)

## Step 3 — Ask vision models about the images

<img src="../../../docs/figs/image_workflow_step_3.svg" alt="Four-stage image-material analysis workflow with Step 3, ask vision models, highlighted." width="100%">

Run the next three cells to ask **BLIP**, an image-captioning model, for one guided description. This is a qualitative sanity check before batch analysis: compare its words with your own first impression. A caption is a model's description, not a verified material inventory—notice both what it names and what it ignores.

**Pause and compare:** Does BLIP name a material directly, use a broader visual cue such as “modern facade,” or make a claim you would challenge from the image alone?

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torch

In [ ]:
# Load processor and model
model_id = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(model_id)
model = BlipForConditionalGeneration.from_pretrained(
    model_id,
    cache_dir=str(MODELS_DIR)
)
model.eval()

In [ ]:
raw_image = Image.open(GENERATED_IMAGES_DIR / "building_exterior_003.png").convert('RGB')

inputs = processor(images=raw_image, text="The facade material of the builiding is", return_tensors="pt")
# inputs = processor(images=raw_image, return_tensors="pt")

with torch.no_grad():
    output = model.generate(**inputs)

caption = processor.decode(output[0], skip_special_tokens=True)
display(raw_image)
print("Generated Caption:", caption)

### Ask the same material question for every image

Now ask **Qwen2.5-VL** the same question—“What is the main facade material?”—for every generated image. Unlike BLIP's single-image caption, this gives the batch a common question and a comparable answer format. The next cells match each answer against a chosen material-word list and save image filename, answer, and matched words in `output/results.csv`.

This is **candidate-label extraction**, not recognition ground truth. For example, the word `metal` may match an answer that is vague, `steel` may be missed if it is not used, and a visible material outside the vocabulary cannot appear in the table.

In [ ]:
# Initialize model
keywords_list = [
    "glass", "concrete", "brick", "metal", "wood", "stone", "ceramic",
    "steel", "aluminum", "marble", "plaster", "cladding", "tile",
    "granite", "copper", "composite"
] # interested material for matching the extracted captions

processor = VisionQAProcessor(
    model_name="Qwen/Qwen2.5-VL-3B-Instruct",
    cache_dir=str(MODELS_DIR),
    keywords_list=keywords_list
)

In [ ]:
# Caption the generated images
prompt = "What is the main facade material of the building in this image?"

df_test = processor.batch_image_qa(
    image_folder=str(TEST_IMAGES_DIR),
    prompt=prompt,
    save_csv_path=str(OUTPUT_DIR / "test_results.csv")
)

In [ ]:
result_df = processor.batch_image_qa(
    image_folder=str(GENERATED_IMAGES_DIR),
    prompt=prompt,
    save_csv_path=str(OUTPUT_DIR / "results.csv")
)

In [ ]:
results_df = pd.read_csv(OUTPUT_DIR / "results.csv")

# show the first few lines
for index, result in results_df.head().iterrows():
    print("Answer:", result['answer'])
    print("Keywords:", result['materials'])
    print("-" * 40)

## Step 4 — View label patterns

<img src="../../../docs/figs/image_workflow_step_4.svg" alt="Four-stage image-material analysis workflow with Step 4, view patterns in labels, highlighted." width="100%">

The final two visualisations answer different questions about the matched labels. Before running them, remember what is being counted: words matched from model answers, not materials measured by an expert.

### How to read the charts

First run the **frequency chart** cell. A bar's height is the number of images whose Qwen-VL answer matched that word in the chosen vocabulary. A taller `glass` bar means this pipeline produced more `glass` matches; it does not mean glass is objectively more present in the images, much less in real architecture.

Then run the **co-occurrence matrix** cell. Each row and column is one of the ten most frequent matched words. A diagonal value counts images containing that word. An off-diagonal value counts images whose answer matched **both** words—such as `glass` and `metal`. Read a dark or large off-diagonal cell as “these labels often appeared together in this pipeline,” not “these materials necessarily occur together in the generated scene.”

In [ ]:
# Check what's the most detected material from the AI-generated images
save_path = OUTPUT_DIR / "keywords_freq.png"
freqs = viz_utils.viz_keywords_freq(results_df, column="materials", top_k=10, save_path=save_path)

## Next: which labels appear together?

The frequency plot answers **which matched labels are most common?** The next cell asks a different question: **which matched labels were assigned to the same image?** It creates a co-occurrence matrix for the ten most frequent labels.

Read it in two steps: choose a row label and a column label; then read their intersection as the number of images where both words were matched. The diagonal is a label's own count. If `glass`–`metal` is high, investigate several underlying answers and images: it may reflect a recurring visual association, a broad answer style, or the vocabulary-matching rule.

In [ ]:
# Display the Co-occurrence of each material
save_path = OUTPUT_DIR / "cooccurrence.png"
co_df = viz_utils.plot_cooccurrence_heatmap(results_df, top_k=10, save_path=save_path)

## Application matrix: what would make an extension meaningful?

The current experiment is a controlled baseline. Each extension needs its own reference standard; generated-image labels alone are not ground truth.

| Extension | Dataset needed | Annotation/reference standard | Reused functions/variables | Why it is meaningful |
| --- | --- | --- | --- | --- |
| Prompt-condition comparison | Balanced generated sets for each prompt condition | Pre-registered prompt changes, fixed seeds, and equal image counts | `client`, `prompt`, `seed`, `results_df`, `viz_keywords_freq` | Tests whether a specified prompt change shifts candidate-label patterns |
| Material recognition | Expert-labelled facade-image dataset | Image-level material labels plus an adjudication rule | `VisionQAProcessor.extract_keywords`, `keywords_list`, `cooccurrence` logic | Compares model labels with a reference instead of treating generated images as truth |
| Representation audit | Balanced prompts and images covering the groups of interest | Human coding guide, reviewer agreement, and group-comparison protocol | `model_id`, `num_images`, `results.csv`, `seed` | Makes an association claim inspectable and bounded |
| Robustness check | Same images processed by recorded model versions | Version manifest and repeated-seed or cross-model agreement | `cache_dir`, `prompt`, `results_df`, chart code | Separates a pipeline artefact from a stable visual-language pattern |

## Wrap-up: from a prompt to an investigable question

In this tutorial, you followed one complete multimodal chain: a neutral written prompt became a controlled set of images; vision models turned those images back into captions and candidate labels; and simple charts made recurring model-label patterns visible. The value of the workflow is not that one chart proves a bias. It is that it turns a broad concern about representation into a traceable question with inspectable images, prompts, answers, and matching rules.

### Extend this tutorial

- **Compare prompt conditions.** Change one carefully chosen element—for example, building type or urban context—while keeping the seed strategy and question fixed. This lets you compare label patterns across prompt conditions.
- **Evaluate material recognition.** Replace generated images with a labelled facade-image set, then compare vision-model labels with expert or annotated reference labels.
- **Design a stronger association audit.** Specify a hypothesis, a balanced prompt sample, a vocabulary or coding protocol, and a human-review plan before comparing groups.

One prompt and 50 images are not a representative sample. Bias can enter through both the image generator and the vision model; a fixed vocabulary can miss labels; and versions or hardware can change output. Treat this activity as a reusable way to formulate and inspect a question—not as evidence for a final claim about materials or bias.